# LIBERO-Plus, family 1: OpenVLA-OFT (L1-regression chunks)
**GPU: A100** (L4 works, slower). Setup ~20 min (includes 6.4 GB LIBERO-Plus assets and the 15 GB checkpoint).
Full run: ~210 perturbed LIBERO-10 tasks, 1 episode each, about 1.5 to 2 h. Resumable: rerun the last cells after a disconnect.
Paste back the printed table of the last cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline matplotlib backend crashes every venv python
os.environ['PYTHONUNBUFFERED'] = '1'   # run cells are piped into a log on Drive; without this their output arrives in bursts
os.makedirs('/content/drive/MyDrive/restoring_organs/plus/logs', exist_ok=True)
os.makedirs('/content/logs', exist_ok=True)
print('outputs ->', '/content/drive/MyDrive/restoring_organs/plus')
# Drive uploads a file only after it is CLOSED, so a log held open by a running cell never arrives. Run cells log to
# /content/logs; this thread rewrites the last 64 KB of each log on Drive once a minute as a fresh, closed file.
import builtins, glob, threading, time
def _mirror_logs():
    while True:
        for p in glob.glob('/content/logs/*.log'):
            try:
                with open(p, 'rb') as f:
                    f.seek(max(0, os.path.getsize(p) - 65536))
                    tail = f.read()
                with open('/content/drive/MyDrive/restoring_organs/plus/logs/' + os.path.basename(p)[:-4] + '.' + os.uname().nodename[-6:] + '.log', 'wb') as f:
                    f.write(tail)
            except Exception:
                pass
        time.sleep(60)
if not getattr(builtins, '_log_mirror_on', False):
    builtins._log_mirror_on = True
    threading.Thread(target=_mirror_logs, daemon=True).start()
# Hugging Face login from the Colab secret HF_TOKEN (key icon, left sidebar), if you added one; optional here
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('Hugging Face login ok')
except Exception as e:
    print('no HF_TOKEN secret, continuing without Hugging Face login:', type(e).__name__)

In [ ]:
import os; os.makedirs('/content/scripts', exist_ok=True); print('ok')

In [ ]:
%%writefile /content/scripts/plus_common.py
"""Shared by every LIBERO-Plus notebook: one deterministic task subset (so all five policy families see
exactly the same perturbed tasks), perturbation categories parsed from task names, and the summary table."""
import json
import math
import os
import random
import re
from collections import Counter, defaultdict

# LIBERO-Plus task names, checked on the real suite: token perturbations carry their own tag; camera and robot
# initial state are encoded as values, e.g. ..._view_0_0_100_0_0_initstate_0 is the unperturbed setting.
CATS = [("sensor_noise", r"_noise_\d+"), ("language", r"_language_\d+"), ("light", r"_light_\d+"),
        ("background", r"_table_\d+|_tb_\d+"), ("layout", r"_add_\d+|_level")]
DEFAULT_VIEW = ("0", "0", "100", "0", "0")
# Every run script reads the suite from the environment, so one variable switches a whole notebook:
#   LIBERO_SUITE=libero_spatial | libero_object | libero_goal | libero_10 (default; nothing changes when it is unset)
SUITE = os.environ.get("LIBERO_SUITE", "libero_10")
MAX_STEPS = {"libero_spatial": 220, "libero_object": 280, "libero_goal": 300, "libero_10": 520}     # OFT's own horizons
OFT_CKPT = {s: "moojink/openvla-7b-oft-finetuned-" + s.replace("_", "-") for s in MAX_STEPS}


def categorize(name):
    for cat, pat in CATS:
        if re.search(pat, name):
            return cat
    m = re.search(r"_view_(\d+)_(\d+)_(\d+)_(\d+)_(\d+)", name)
    if m and m.groups() != DEFAULT_VIEW:
        return "camera"
    s = re.search(r"_initstate_(\d+)", name)
    if s and s.group(1) != "0":
        return "robot_init"
    return "other"


def build_subset(suite_name="libero_10", per_cat=30, seed=0):
    from libero.libero import benchmark
    suite = benchmark.get_benchmark_dict()[suite_name]()
    names = [suite.get_task(i).name for i in range(suite.n_tasks)]
    langs = [suite.get_task(i).language for i in range(suite.n_tasks)]
    by = defaultdict(list)
    for i, n in enumerate(names):
        by[categorize(n)].append(i)
    rng = random.Random(seed)
    pick = []
    for cat in sorted(by):
        ids = sorted(by[cat])
        rng.shuffle(ids)
        pick += ids[:per_cat]
    pick.sort()
    tokens = Counter(tok for n in names for tok in re.findall(r"_([a-z]+)_\d+", n))
    return {"suite": suite_name, "n_tasks": suite.n_tasks, "per_cat": per_cat, "seed": seed,
            "counts": {c: len(v) for c, v in sorted(by.items())},
            "other_examples": [names[i] for i in by.get("other", [])[:5]],
            "name_tokens": dict(tokens.most_common(20)),
            "tasks": [{"task_id": i, "name": names[i], "language": langs[i], "category": categorize(names[i])}
                      for i in pick]}


def load_or_build(path, per_cat=30):
    if os.path.exists(path):
        return json.load(open(path))
    sub = build_subset(suite_name=SUITE, per_cat=per_cat)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    json.dump(sub, open(path, "w"), indent=1)
    return sub


def load_rows(path):
    rows = []
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass
    return rows


def append_row(path, row):
    """Append one result row and CLOSE the file. Drive uploads a file only when it is closed: a runner that keeps its
    results file open for a whole cell has nothing on Drive until the cell ends, and a killed runtime loses all of it."""
    with open(path, "a") as f:
        f.write(json.dumps(row, default=float) + "\n")


def load_shards(path):
    """Rows of `<stem>.jsonl` plus every per-session shard `<stem>.<shard>.jsonl` next to it, one row per
    (task, condition, arm, rep). Two Colab sessions that append to ONE Drive file overwrite each other (each uploads its
    own whole copy, the last upload wins; file 8 lost 576 rows that way on 2026-09-20), so every session writes its own
    shard and readers merge. A measured row beats one recovered from a log (`recovered`), which lacks the detail fields."""
    import glob
    best = {}
    # 0 = measured, 1 = measured before the shading gate (re-queued by the runner, its re-run lives in a later shard),
    # 2 = rebuilt from a log. The lowest rank wins; on a tie the first file does.
    rank = lambda r: 2 if r.get("recovered") else int(bool(r.get("photo_applied")) and "photo_shading" not in r)
    for p in [path] + sorted(glob.glob(path[:-len(".jsonl")] + ".*.jsonl")):
        for r in load_rows(p):
            k = (r.get("policy"), r["task_id"], r.get("cond"), r["arm"], r.get("rep", 0))
            if k not in best or rank(r) < rank(best[k]):
                best[k] = r
    return list(best.values())


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)


def summarize(rows, label=""):
    by = defaultdict(list)
    for r in rows:
        by[r["category"]].append(bool(r["success"]))
    out = {}
    print(f"\n{label}  ({len(rows)} episodes)")
    print(f"{'category':<14}{'n':>5}{'success':>9}{'95% CI':>16}")
    for cat in sorted(by) + ["ALL"]:
        v = [s for r in by.values() for s in r] if cat == "ALL" else by[cat]
        k, n = sum(v), len(v)
        lo, hi = wilson(k, n)
        out[cat] = {"n": n, "success": k / max(1, n), "ci": [lo, hi]}
        print(f"{cat:<14}{n:>5}{k / max(1, n):>9.3f}   [{lo:.2f}, {hi:.2f}]")
    return out

In [ ]:
%%writefile /content/scripts/eval_oft.py
"""OpenVLA-OFT on the LIBERO-Plus subset, using OFT's own run_episode. Run from the openvla-oft repo root."""
import argparse
import gc
import json
import os
import sys

os.environ.setdefault("MUJOCO_GL", "egl")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from plus_common import load_rows, summarize  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--subset", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--ckpt", default="moojink/openvla-7b-oft-finetuned-libero-10")
    ap.add_argument("--limit", type=int, default=0)
    a = ap.parse_args()

    from libero.libero import benchmark
    from experiments.robot.libero.libero_utils import get_libero_env
    from experiments.robot.libero.run_libero_eval import GenerateConfig, initialize_model, run_episode
    from experiments.robot.robot_utils import get_image_resize_size, set_seed_everywhere

    tasks = json.load(open(a.subset))["tasks"]
    tasks = tasks[:a.limit] if a.limit else tasks
    cfg = GenerateConfig(pretrained_checkpoint=a.ckpt, task_suite_name="libero_10", center_crop=True)
    set_seed_everywhere(cfg.seed)
    model, action_head, proprio_projector, noisy_action_projector, processor = initialize_model(cfg)
    resize_size = get_image_resize_size(cfg)
    suite = benchmark.get_benchmark_dict()["libero_10"]()
    done = {r["task_id"] for r in load_rows(a.out)}
    os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
    with open(a.out, "a") as f:
        for t in tasks:
            if t["task_id"] in done:
                continue
            task = suite.get_task(t["task_id"])
            init = suite.get_task_init_states(t["task_id"])
            env, desc = get_libero_env(task, cfg.model_family, resolution=cfg.env_img_res)
            success, _ = run_episode(cfg, env, desc, model, resize_size, processor, action_head,
                                     proprio_projector, noisy_action_projector, init[0])
            env.close()
            del env
            gc.collect()
            row = dict(family="openvla_oft", **t, success=bool(success))
            f.write(json.dumps(row) + "\n")
            f.flush()
            print(f"{t['task_id']:5d} {t['category']:<12} success={bool(success)}", flush=True)
    summarize([r for r in load_rows(a.out)], "OpenVLA-OFT")


if __name__ == "__main__":
    main()

In [ ]:
%%bash
set -e
pip -q install uv
[ -x /content/venv_oft/bin/python ] || uv venv -q /content/venv_oft -p 3.10
cd /content
[ -d openvla-oft ] || git clone -q https://github.com/moojink/openvla-oft.git
uv pip install -q --python /content/venv_oft/bin/python -e /content/openvla-oft

apt-get -qq update > /dev/null
apt-get -qq install -y libexpat1 libfontconfig1-dev libmagickwand-dev unzip > /dev/null
cd /content
[ -d LIBERO-plus ] || git clone -q --depth 1 https://github.com/sylvestf/LIBERO-plus.git
uv pip install -q --python /content/venv_oft/bin/python -e /content/LIBERO-plus 
uv pip install -q --python /content/venv_oft/bin/python wand scikit-image
# LIBERO's setup.py does not register the top-level package, so the editable install alone is not importable
SITE=$(/content/venv_oft/bin/python -c "import sysconfig; print(sysconfig.get_paths()['purelib'])")
echo /content/LIBERO-plus > "$SITE/libero_plus.pth"
/content/venv_oft/bin/python -c "import importlib.util as u; assert u.find_spec('libero') is not None, 'libero still not importable'; print('libero importable')"
if [ ! -f /content/LIBERO-plus/.assets_ok ]; then
  echo "downloading LIBERO-Plus assets (6.4 GB)..."
  # Hugging Face answers a rate-limited or interrupted download with an error page or a short file; plain `curl -s`
  # saved that as the zip and unzip then failed with no message (2026-09-19). Fail on HTTP errors, resume, retry, send
  # the HF_TOKEN when there is one (anonymous downloads are limited first), and test the archive before unpacking.
  ZIP=/content/lp_assets.zip
  URL=https://huggingface.co/datasets/sii-research/LIBERO_plus_assets/resolve/main/assets.zip
  ok=0
  for try in 1 2 3 4 5 6; do
    if [ -f $ZIP ] && unzip -tq $ZIP > /dev/null 2>&1; then ok=1; break; fi
    size=$(stat -c %s $ZIP 2>/dev/null || echo 0)
    if [ $try -gt 1 ]; then echo "asset download attempt $((try-1)) incomplete ($size bytes), retrying in 30 s"; sleep 30; fi
    if [ "$size" -lt 1000000 ]; then rm -f $ZIP; fi                 # an error page cannot be resumed
    curl -fL -C - --retry 3 --retry-delay 15 -sS ${HF_TOKEN:+-H "Authorization: Bearer $HF_TOKEN"} -o $ZIP $URL || true
  done
  if [ $ok != 1 ]; then echo "LIBERO-Plus assets could not be downloaded (Hugging Face rate limit or outage). Wait a few minutes and re-run this cell; it resumes."; exit 1; fi
  rm -rf /content/lp_assets && mkdir -p /content/lp_assets
  unzip -q /content/lp_assets.zip -d /content/lp_assets && rm -f /content/lp_assets.zip
  python3 - <<'EOF'
import os, shutil
root, dst = "/content/lp_assets", "/content/LIBERO-plus/libero/libero/assets"
cands = [os.path.join(d, n) for d, ns, _ in os.walk(root) for n in ns if n == "assets"]
src = min(cands, key=len) if cands else root
os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print("assets:", src, "->", dst, "|", len(os.listdir(dst)), "entries")
EOF
  touch /content/LIBERO-plus/.assets_ok
fi

uv pip install -q --python /content/venv_oft/bin/python -r /content/openvla-oft/experiments/robot/libero/libero_requirements.txt
uv pip install -q --python /content/venv_oft/bin/python "numpy<2" "mujoco==2.3.7"
uv pip install -q --python /content/venv_oft/bin/python "protobuf==4.25.3"
uv pip install -q --python /content/venv_oft/bin/python --no-deps "tensorflow-metadata==1.16.1"
/content/venv_oft/bin/python -c "import torch, robosuite, mujoco; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'mujoco', mujoco.__version__)"

In [ ]:
# LIBERO-Plus asks for its paths on first import; write them up front
import os
root = '/content/LIBERO-plus/libero/libero'
os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
open(os.path.expanduser('~/.libero/config.yaml'), 'w').write(
    f'benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n'
    f'datasets: {root}/../datasets\nassets: {root}/assets\n')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['MPLBACKEND'] = 'Agg'   # Colab's inline backend is not valid inside the venvs
os.environ['PYTHONPATH'] = '/content/LIBERO-plus'   # Colab's system python ignores the .pth file
print(open(os.path.expanduser('~/.libero/config.yaml')).read())

In [ ]:
# the SAME task subset for every family (built once, then read from Drive)
!MUJOCO_GL=egl /content/venv_oft/bin/python -c "import sys; sys.path.insert(0, '/content/scripts'); import plus_common as pc; s = pc.load_or_build('/content/drive/MyDrive/restoring_organs/plus/subset_libero10.json'); print('suite tasks', s['n_tasks']); print('per category', s['counts']); print('name tokens', s['name_tokens']); print('other examples', s['other_examples']); print(len(s['tasks']), 'tasks in subset')"

In [ ]:
# smoke: 2 tasks
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/eval_oft.py --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out /content/drive/MyDrive/restoring_organs/plus/smoke_oft.jsonl --limit 2 2>&1 | tee -a /content/logs/eval_oft.log

In [ ]:
# full run (resumable)
![ -f /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py ] && python3 /content/drive/MyDrive/restoring_organs/plus/live_scripts/_sync.py; cd /content/openvla-oft && MUJOCO_GL=egl /content/venv_oft/bin/python /content/scripts/eval_oft.py --subset /content/drive/MyDrive/restoring_organs/plus/subset_libero10.json --out /content/drive/MyDrive/restoring_organs/plus/rows_oft.jsonl 2>&1 | tee -a /content/logs/eval_oft.log